In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.ml.classification import GBTClassifier

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.feature import StringIndexer

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
2,application_1785763816403_0003,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


In [2]:
ml_df = spark.read.parquet(
    "s3://airline-dataset-2020-2025/GoldM1/ML_DATASET/"
)

print(ml_df.count())
print(len(ml_df.columns))

ml_df.show(5)

39895374
41
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-------+----------+---------+-------------+-----------+---------------+-------------+-----------------+----------------+---------------+--------------------+--------+---------------------------+----------------+-------------+-------------------+--------+-----------------------+-----------------------------+---------------------------+---------------------+------------------+------------------------+----------------------+----------------+----------------+-------------------+------------------------+-----------------------+----------------------+--------------------+------------+-----------------------+----+-----+
|      DestAirportKey|    OriginAirportKey| MarketingAirlineKey|            RouteKey|           FlightKey|FlightDate|Quarter|DayofMonth|DayOfWeek|DepartureHour|ArrivalHour|DeparturePeriod|ArrivalPeriod|PeakHourIndicator|WeekendIndicator|SeasonIndicat

In [3]:
ml_df.columns

['DestAirportKey', 'OriginAirportKey', 'MarketingAirlineKey', 'RouteKey', 'FlightKey', 'FlightDate', 'Quarter', 'DayofMonth', 'DayOfWeek', 'DepartureHour', 'ArrivalHour', 'DeparturePeriod', 'ArrivalPeriod', 'PeakHourIndicator', 'WeekendIndicator', 'SeasonIndicator', 'OperatingAirlineKey', 'Distance', 'ScheduledElapsedTimeMinutes', 'DistanceCategory', 'CodeshareFlag', 'IntraStateRouteFlag', 'ArrDel15', 'AirlineReliabilityScore', 'OriginAirportReliabilityScore', 'DestAirportReliabilityScore', 'RouteReliabilityScore', 'AirlineFlightCount', 'OriginAirportFlightCount', 'DestAirportFlightCount', 'RouteFlightCount', 'RouteAvgDistance', 'RouteAvgElapsedTime', 'RouteHistoricalDelayRate', 'AirlineMonthlyDelayRate', 'OriginMonthlyDelayRate', 'DestMonthlyDelayRate', 'DatasetSplit', 'ReliabilityFeatureScope', 'Year', 'Month']

In [4]:
ml_df.printSchema()

root
 |-- DestAirportKey: string (nullable = true)
 |-- OriginAirportKey: string (nullable = true)
 |-- MarketingAirlineKey: string (nullable = true)
 |-- RouteKey: string (nullable = true)
 |-- FlightKey: string (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepartureHour: integer (nullable = true)
 |-- ArrivalHour: integer (nullable = true)
 |-- DeparturePeriod: string (nullable = true)
 |-- ArrivalPeriod: string (nullable = true)
 |-- PeakHourIndicator: integer (nullable = true)
 |-- WeekendIndicator: integer (nullable = true)
 |-- SeasonIndicator: string (nullable = true)
 |-- OperatingAirlineKey: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- ScheduledElapsedTimeMinutes: integer (nullable = true)
 |-- DistanceCategory: string (nullable = true)
 |-- CodeshareFlag: integer (nullable = true)
 |-- IntraStateRouteFlag: inte

Step 2 Split

In [5]:
train_df = ml_df.filter(F.col("DatasetSplit")=="Train")

validation_df = ml_df.filter(F.col("DatasetSplit")=="Validation")

test_df = ml_df.filter(F.col("DatasetSplit")=="Test")

In [6]:
#Remove Unnecessary Columns
drop_cols = [

"FlightKey",
"FlightDate",
"DatasetSplit",
"ReliabilityFeatureScope"

]

train_df = train_df.drop(*drop_cols)
validation_df = validation_df.drop(*drop_cols)
test_df = test_df.drop(*drop_cols)

In [7]:
#Categorical Columns
categorical_cols = [

"SeasonIndicator",

"DeparturePeriod",
"ArrivalPeriod",

"DistanceCategory",

"MarketingAirlineKey",
"OperatingAirlineKey",

"OriginAirportKey",
"DestAirportKey",

"RouteKey"

]

In [8]:
#Step 5 Numeric Columns
numeric_cols = [

"Year",
"Quarter",
"Month",

"DayofMonth",
"DayOfWeek",

"DepartureHour",
"ArrivalHour",

"PeakHourIndicator",
"WeekendIndicator",

"Distance",
"ScheduledElapsedTimeMinutes",

"CodeshareFlag",
"IntraStateRouteFlag",

"AirlineReliabilityScore",
"OriginAirportReliabilityScore",
"DestAirportReliabilityScore",
"RouteReliabilityScore",

"AirlineFlightCount",
"OriginAirportFlightCount",
"DestAirportFlightCount",

"RouteFlightCount",

"RouteAvgDistance",
"RouteAvgElapsedTime",

"RouteHistoricalDelayRate",

"AirlineMonthlyDelayRate",
"OriginMonthlyDelayRate",
"DestMonthlyDelayRate"

]

In [9]:
#Step 6 String Indexers
indexers = [

StringIndexer(

inputCol=c,
outputCol=c+"_idx",
handleInvalid="keep"

)

for c in categorical_cols

]

In [10]:
#Step 7 OneHotEncoder
from pyspark.ml.feature import OneHotEncoder

encoders = [

OneHotEncoder(

inputCol=c+"_idx",
outputCol=c+"_Vec"

)

for c in categorical_cols

]

In [11]:
#Step 8 Feature Vector
feature_columns = numeric_cols + [

c+"_Vec"

for c in categorical_cols

]

In [12]:
#Step 9 VectorAssembler
assembler = VectorAssembler(

inputCols=feature_columns,

outputCol="features"

)

In [13]:
#Step 10 Label Indexer
train_df = train_df.withColumn(

"label",

F.col("ArrDel15").cast("double")

)

validation_df = validation_df.withColumn(

"label",

F.col("ArrDel15").cast("double")

)

test_df = test_df.withColumn(

"label",

F.col("ArrDel15").cast("double")

)

In [14]:
#Step 11 Model
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(

labelCol="label",
featuresCol="features",

numTrees=20,

maxDepth=8,

seed=42

)

In [15]:
#Step 12 Pipeline
pipeline = Pipeline(

stages=

indexers

+

encoders

+

[assembler]

+

[rf]

)

In [16]:
#Step 13 Train
model = pipeline.fit(train_df)

An error occurred while calling o130.fit.
: org.apache.spark.SparkException: Job 18 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$cleanUpAfterSchedulerStop$1.apply(DAGScheduler.scala:972)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$cleanUpAfterSchedulerStop$1.apply(DAGScheduler.scala:970)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:78)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:970)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:2284)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:84)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:2191)
	at org.apache.spark.SparkContext$$anonfun$stop$6.apply$mcV$sp(SparkContext.scala:1949)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1340)
	at org.apache.spark.SparkContext.stop(SparkContext.scala:1948)
	at org.apache.spark.scheduler.cluster.Yar

In [ ]:
#Step 14 Validation Prediction
validation_predictions = model.transform(validation_df)

In [ ]:
#Step 15 Test Prediction
test_predictions = model.transform(test_df)

In [ ]:
#Step 16 ROC
from pyspark.ml.evaluation import BinaryClassificationEvaluator

binary_eval = BinaryClassificationEvaluator(

labelCol="label",

metricName="areaUnderROC"

)

roc = binary_eval.evaluate(validation_predictions)

print(roc)

In [ ]:
#Step 17 Accuracy
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(

labelCol="label",

predictionCol="prediction",

metricName="accuracy"

)

print(

accuracy.evaluate(validation_predictions)

)

In [ ]:
#Step 18 Precision
precision = MulticlassClassificationEvaluator(

labelCol="label",

predictionCol="prediction",

metricName="weightedPrecision"

)

print(

precision.evaluate(validation_predictions)

)

In [ ]:
#Step 19 Recall
recall = MulticlassClassificationEvaluator(

labelCol="label",

predictionCol="prediction",

metricName="weightedRecall"

)

print(

recall.evaluate(validation_predictions)

)

In [ ]:
#Step 20 F1
f1 = MulticlassClassificationEvaluator(

labelCol="label",

predictionCol="prediction",

metricName="f1"

)

print(

f1.evaluate(validation_predictions)

)

In [ ]:
#Step 21 Confusion Matrix
validation_predictions.groupBy(

"label",

"prediction"

).count().show()

In [ ]:
#Step 22 Feature Importance
rf_model = model.stages[-1]

importance = rf_model.featureImportances

print(importance)

In [ ]:
#Step 23 Save Model
model.write().overwrite().save(

"s3://airline-dataset-2020-2025/GoldM1/MODEL/random_forest"

)

In [ ]:
#Step 24 Load Model
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load(

"s3://airline-dataset-2020-2025/GoldM1/MODEL/random_forest"

)